In [13]:
import re
import importlib
import requests
import datetime
import json
from pathlib import Path

from tqdm import tqdm
import pandas as pd
import numpy as np

import lib


importlib.reload(lib)

log = lib.getLogger('goodreads_dataset')
working_directory = Path().cwd()
export_clean_directory = working_directory / 'clean'

In [14]:
path = '/Users/christophermagno/PycharmProjects/MagnoDataAnalystPortfolio/datasets/goodreads/goodreads_library_export.csv'

In [15]:
df = lib.read_data(path)

PortfolioLogger.lib.tools: INFO: Encoding: utf-8
PortfolioLogger.lib.tools: INFO: Stripping whitespaces from goodreads_library_export.csv
PortfolioLogger.lib.performance: INFO: <function read_data at 0x10bf4eac0> took 0.036 secs to complete.


In [4]:
isbn = df.loc[2, 'ISBN13']

In [48]:
def clean_isbn(isbn):
    return re.sub(r'["=]', '', isbn)

In [17]:
def request_openlibrary_book_data(isbn):
    isbn = clean_isbn(isbn)
    url = f"https://openlibrary.org/isbn/{isbn}.json"
    response = requests.get(url)
    if not response:
        return {}

    response = response.json()

    # Get book "works" metadata (contains subjects)
    works_key = response.get("works", [{}])[0].get("key")
    if not works_key:
        return {}

    works_data = requests.get(f"https://openlibrary.org{works_key}.json").json()
    return works_data

In [7]:
request_openlibrary_book_data(isbn)

{'type': {'key': '/type/work'},
 'title': 'The Will of the Many',
 'authors': [{'author': {'key': '/authors/OL7468631A'},
   'type': {'key': '/type/author_role'}}],
 'key': '/works/OL31088394W',
 'subjects': ['series:Hierarchy', 'high fantasy'],
 'description': "The Catenan Republic—the Hierarchy—may rule the world, but they do not know everything.\r\n\r\nI tell them my name is Vis Telimus. I tell them I was orphaned three years ago, and that only good fortune has got me into their most prestigious school. I tell them that, when I graduate, I will allow my strength and drive—what they call Will—to be leeched away and added to the power of those above me, as everyone must do.\r\n\r\nI tell them that I belong, and they believe me.\r\n\r\nBut the truth is that I have been sent to the Academy to solve a murder. To search for an ancient weapon. To uncover secrets that may tear the Republic apart.\r\n\r\nAnd that I will never cede my Will to the empire that executed my family.\r\n\r\nTo surv

# Data that's useful
- volumeInfo
    - authors
    - title
    - publisher
    - publishDate
    - description
    - pageCount
    - categories [list]
    - maturityRating
    - imagelinks (do i want images?): [dict]
    - saleInfo
        - country
        - retailPrice

In [8]:
def request_googleapi_book_data(isbn):
    isbn = clean_isbn(isbn)
    url = f"https://www.googleapis.com/books/v1/volumes?q=isbn:{isbn}"
    response = requests.get(url).json()
    if not 'items' in response:
        return {}
    return response['items'][0]

In [9]:
request_googleapi_book_data(isbn)

{'kind': 'books#volume',
 'id': 'jlGUEAAAQBAJ',
 'etag': '+G4P0wke6pE',
 'selfLink': 'https://www.googleapis.com/books/v1/volumes/jlGUEAAAQBAJ',
 'volumeInfo': {'title': 'The Will of the Many',
  'authors': ['James Islington'],
  'publisher': 'Simon and Schuster',
  'publishedDate': '2023-05-23',
  'description': 'At the elite Catenan Academy, a young fugitive uncovers layered mysteries and world-changing secrets in this “brilliant and gut-churning masterpiece” (Library Journal, starred review) by the internationally bestselling author of The Licanius Trilogy, James Islington. The Catenan Republic—the Hierarchy—may rule the world now, but they do not know everything. I tell them my name is Vis Telimus. I tell them I was orphaned after a tragic accident three years ago, and that good fortune alone has led to my acceptance into their most prestigious school. I tell them that once I graduate, I will gladly join the rest of civilized society in allowing my strength, my drive, and my focus—

In [19]:
def clean_categories_list(categories, ignore=('collectionID', 'nyt')):
    new_categories = set()
    if not categories:
        return list(new_categories)
    for i in range(len(categories)):
        for category in categories[i].split(', '):
            found = False
            for item in ignore:
                if item in category:
                    found = True
                    break
            if found:
                continue
            new_categories.add(category.title().replace('_', ' '))

    return list(new_categories)


In [32]:
def get_book_data(isbn):
    isbn = clean_isbn(isbn)

    result = {'Categories': []}

    openai_categories_data = request_openlibrary_book_data(isbn).get('subjects', [])
    googleapi_data = request_googleapi_book_data(isbn)
    if googleapi_data:
        result.update(
            {'Published Date': googleapi_data['volumeInfo'].get('publishedDate'),
            'Categories': clean_categories_list(googleapi_data['volumeInfo'].get('categories')),
            'Maturity Rating': googleapi_data['volumeInfo'].get('maturityRating'),
            'Description': googleapi_data['volumeInfo'].get('description'),
            'Country': googleapi_data['saleInfo'].get('country'),
            'Retail Price': googleapi_data['saleInfo'].get('retailPrice', {}).get('amount'),
            'Currency': googleapi_data['saleInfo'].get('retailPrice', {}).get('currencyCode')}
        )
    if openai_categories_data:
        result['Categories'].extend(clean_categories_list(openai_categories_data))
    result['Categories'] = sorted(set((result['Categories'])))

    return result


Sample request to see what data I get

In [12]:
data = get_book_data('="9781635575583"')
data

{'Categories': ['Action & Adventure',
  'Adaptations',
  'Blessing And Cursing',
  "Children'S Fiction",
  'Cours Et Courtisans',
  'Courts And Courtiers',
  'Etc. Pour La Jeunesse',
  'Fairies',
  'Fantasy',
  'Fantasy & Magic',
  'Fantasy Fiction',
  'Fiction',
  'Fées',
  'General',
  'Love & Romance',
  'Magic',
  'New York Times Bestseller',
  'Nouvelles',
  'Romans',
  'Series:A Court Of Thorns And Roses'],
 'Published Date': '2020-06-02',
 'Maturity Rating': 'NOT_MATURE',
 'Description': "Though Feyre now has the powers of the High Fae, her heart remains human, but as she navigates the feared Night Court's dark web of politics, passion, and dazzling power, a greater evil looms--and she might be key to stopping it.",
 'Country': 'US',
 'Retail Price': None}

# Add the extra data into the DataFrame

In [13]:
df.columns

Index(['Book Id', 'Title', 'Author', 'Author l-f', 'Additional Authors',
       'ISBN', 'ISBN13', 'My Rating', 'Average Rating', 'Publisher', 'Binding',
       'Number of Pages', 'Year Published', 'Original Publication Year',
       'Date Read', 'Date Added', 'Bookshelves', 'Bookshelves with positions',
       'Exclusive Shelf', 'My Review', 'Spoiler', 'Private Notes',
       'Read Count', 'Owned Copies'],
      dtype='object')

In [34]:
categories = []
concat_df = pd.DataFrame(df['Book Id'])
df_categories = pd.DataFrame()
for idx, row in tqdm(df.iterrows(), total=len(df)):
    isbn = clean_isbn(row['ISBN13'])
    book_data = get_book_data(isbn)
    if book_data:
        for key, value in book_data.items():
            if key == 'Categories':
                for category in book_data['Categories']:
                    categories.append(
                        {'Book Id': row['Book Id'], 'Category': category}
                    )
                continue
            concat_df.at[idx, key] = value

concat_df

100%|██████████| 237/237 [03:04<00:00,  1.29it/s]


,Book Id,Published Date,Maturity Rating,Description,Country,Retail Price,Currency
0,18400112,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None
1,62047984,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None
2,58416952,2023-05-23,NOT_MATURE,"At the elite Catenan Academy, a young fugitive...",US,16.99,USD
3,20886354,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None
4,58778536,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None
...,...,...,...,...,...,...,...
232,40389527,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None
233,38389488,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None
234,39863488,2019-02-05,NOT_MATURE,"INSTANT NEW YORK TIMES BESTSELLER ""I love Jane...",US,None,None
235,43848929,2019-09-10,NOT_MATURE,"Malcolm Gladwell, host of the podcast Revision...",US,None,None


In [35]:
df_cp1 = df.merge(concat_df, how='left', on='Book Id')

In [36]:
df_cp1

,Book Id,Title,Author,Author l-f,Additional Authors,ISBN,ISBN13,My Rating,Average Rating,Publisher,...,Spoiler,Private Notes,Read Count,Owned Copies,Published Date,Maturity Rating,Description,Country,Retail Price,Currency
0,18400112,"The Devil Wears Scrubs (Dr. Jane McGill, #1)",Freida McFadden,"McFadden, Freida",NaN,"=""""","=""""",3,3.44,Hollywood Upstairs Publishing,...,NaN,NaN,1,0,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None
1,62047984,Yellowface,R.F. Kuang,"Kuang, R.F.",NaN,"=""""","=""""",4,3.73,William Morrow,...,NaN,NaN,1,0,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None
2,58416952,"The Will of the Many (Hierarchy, #1)",James Islington,"Islington, James",NaN,"=""1982141190""","=""9781982141196""",0,4.60,Gallery / Saga Press,...,NaN,NaN,0,0,2023-05-23,NOT_MATURE,"At the elite Catenan Academy, a young fugitive...",US,16.99,USD
3,20886354,"Skin Deep (Legion, #2)",Brandon Sanderson,"Sanderson, Brandon",Jon Foster,"=""""","=""""",4,4.12,Subterranean Press,...,NaN,NaN,1,0,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None
4,58778536,Do Not Disturb,Freida McFadden,"McFadden, Freida",NaN,"=""""","=""""",3,3.90,Hollywood Upstairs Publishing,...,NaN,NaN,1,0,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
232,40389527,The Woman in the Window,A.J. Finn,"Finn, A.J.",NaN,"=""""","=""""",0,3.94,William Morrow,...,NaN,NaN,1,0,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None
233,38389488,The Reckoning,John Grisham,"Grisham, John",NaN,"=""""","=""""",0,3.75,Doubleday,...,NaN,NaN,1,0,2025-10-30,NOT_MATURE,Have you ever wondered why that 13-digit numbe...,US,None,None
234,39863488,The Lost Man,Jane Harper,"Harper, Jane",NaN,"=""1250105684""","=""9781250105684""",0,4.16,Flatiron Books,...,NaN,NaN,1,0,2019-02-05,NOT_MATURE,"INSTANT NEW YORK TIMES BESTSELLER ""I love Jane...",US,None,None
235,43848929,Talking to Strangers: What We Should Know Abou...,Malcolm Gladwell,"Gladwell, Malcolm",NaN,"=""0316478520""","=""9780316478526""",0,4.00,"Little, Brown and Company",...,NaN,NaN,1,0,2019-09-10,NOT_MATURE,"Malcolm Gladwell, host of the podcast Revision...",US,None,None


In [37]:
df_categories = pd.DataFrame(categories)

In [38]:
df_categories['Category'].value_counts()

Category
Fiction                        129
General                         80
New York Times Bestseller       77
Language Arts & Disciplines     74
Fantasy                         49
                              ... 
Selenium                         1
Robots In Fiction                1
Robotics                         1
Robopsychology                   1
Trust                            1
Name: count, Length: 2032, dtype: int64

# Cleaning

In [40]:
df_cp1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 237 entries, 0 to 236
Data columns (total 30 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Book Id                     237 non-null    int64  
 1   Title                       237 non-null    object 
 2   Author                      237 non-null    object 
 3   Author l-f                  237 non-null    object 
 4   Additional Authors          55 non-null     object 
 5   ISBN                        237 non-null    object 
 6   ISBN13                      237 non-null    object 
 7   My Rating                   237 non-null    int64  
 8   Average Rating              237 non-null    float64
 9   Publisher                   233 non-null    object 
 10  Binding                     237 non-null    object 
 11  Number of Pages             236 non-null    float64
 12  Year Published              236 non-null    float64
 13  Original Publication Year   235 non

In [47]:
for col, series in df_cp1[['ISBN', 'ISBN13']].items():
    df_cp1[col] = df_cp1[col].apply(clean_isbn)

In [60]:
df_cp1['Additional Authors'], _ = lib.fillnull(df_cp1['Additional Authors'], 'None')
lib.error_rate(df_cp1['Additional Authors'])

In [88]:
df_cp1['Number of Pages'], _ = lib.fillnull(df_cp1['Number of Pages'], 0)
df_cp1['Number of Pages'] = df_cp1['Number of Pages'].astype(int)
lib.error_rate(df_cp1['Number of Pages'])

0.0

In [64]:
df_cp1 = df_cp1.drop('Year Published', axis=1)

In [68]:
df_cp1['Original Publication Year'], _ = lib.fillnull(df_cp1['Original Publication Year'], 0)
df_cp1['Original Publication Year'] = df_cp1['Original Publication Year'].astype(int)

In [81]:
df_cp1['Date Read'] = pd.to_datetime(df_cp1['Date Read'])
df_cp1['Date Added'] = pd.to_datetime(df_cp1['Date Added'])
df_cp1['Published Date'] = pd.to_datetime(df_cp1['Published Date'], format='mixed')

In [86]:
df_cp1['Retail Price'] = df_cp1['Retail Price'].astype(float)

In [89]:
df_cp1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 237 entries, 0 to 236
Data columns (total 29 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   Book Id                     237 non-null    int64         
 1   Title                       237 non-null    object        
 2   Author                      237 non-null    object        
 3   Author l-f                  237 non-null    object        
 4   Additional Authors          237 non-null    object        
 5   ISBN                        237 non-null    object        
 6   ISBN13                      237 non-null    object        
 7   My Rating                   237 non-null    int64         
 8   Average Rating              237 non-null    float64       
 9   Publisher                   233 non-null    object        
 10  Binding                     237 non-null    object        
 11  Number of Pages             237 non-null    int64         

# Export Data

In [92]:
lib.export_data(export_clean_directory / 'my_books_data.csv', df_cp1)
lib.export_data(export_clean_directory / 'my_books_categories.csv', df_categories)

PortfolioLogger.lib.tools: INFO: Exporting 6873 elements (0.62 MB) to /Users/christophermagno/PycharmProjects/MagnoDataAnalystPortfolio/datasets/goodreads/goodreads_dataset/clean/my_books_data_CLEAN.csv
PortfolioLogger.lib.performance: INFO: <function export_data at 0x10bc6aac0> took 0.007 secs to complete.
PortfolioLogger.lib.tools: INFO: Exporting 7540 elements (0.27 MB) to /Users/christophermagno/PycharmProjects/MagnoDataAnalystPortfolio/datasets/goodreads/goodreads_dataset/clean/my_books_categories_CLEAN.csv
PortfolioLogger.lib.performance: INFO: <function export_data at 0x10bc6aac0> took 0.004 secs to complete.


PosixPath('/Users/christophermagno/PycharmProjects/MagnoDataAnalystPortfolio/datasets/goodreads/goodreads_dataset/clean/my_books_categories_CLEAN.csv')

In [18]:
import pandas as pd
import lib
dates = pd.date_range(df['Date Added'].min(), df['Date Added'].max(), freq='D')
dates

DatetimeIndex(['2023-12-02', '2023-12-03', '2023-12-04', '2023-12-05',
               '2023-12-06', '2023-12-07', '2023-12-08', '2023-12-09',
               '2023-12-10', '2023-12-11',
               ...
               '2025-11-22', '2025-11-23', '2025-11-24', '2025-11-25',
               '2025-11-26', '2025-11-27', '2025-11-28', '2025-11-29',
               '2025-11-30', '2025-12-01'],
              dtype='datetime64[ns]', length=731, freq='D')

In [6]:
df_dates = pd.DataFrame(dates, columns=['Date'])
df_dates

,Date
0,2020-01-01
1,2020-01-02
2,2020-01-03
3,2020-01-04
4,2020-01-05
...,...
2187,2025-12-27
2188,2025-12-28
2189,2025-12-29
2190,2025-12-30


In [28]:
years = []
for idx, row in df_dates.iterrows():
    years.append(df_dates.loc[idx, 'Date'].year)
years

[2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,
 2020,

In [29]:
df_dates['Year'] = pd.Series(years)

In [30]:
lib.export_data(export_clean_directory / 'my_books_dates.csv', df_dates)

PortfolioLogger.lib.tools: INFO: Exporting 4384 elements (0.03 MB) to /Users/christophermagno/PycharmProjects/MagnoDataAnalystPortfolio/datasets/goodreads/clean/my_books_dates_CLEAN.csv
PortfolioLogger.lib.performance: INFO: <function export_data at 0x10bf4ec00> took 0.007 secs to complete.


PosixPath('/Users/christophermagno/PycharmProjects/MagnoDataAnalystPortfolio/datasets/goodreads/clean/my_books_dates_CLEAN.csv')

In [31]:
df_dates

,Date,Year
0,2020-01-01,2020
1,2020-01-02,2020
2,2020-01-03,2020
3,2020-01-04,2020
4,2020-01-05,2020
...,...,...
2187,2025-12-27,2025
2188,2025-12-28,2025
2189,2025-12-29,2025
2190,2025-12-30,2025
